In [ ]:
import requests
import json
import base64

# ========== 核心 Agent ==========
class ERPAgent:
    def __init__(self):
        self.round = 1
        self.history = []
        self.competitor_history = []  # 记录对手广告费
        self.data = {
            "现金": 100,
            "固定资产": 50,
            "库存": 0,
            "贷款": 0,
            "所有者权益": 100,
            "研发进度": {"P1": "完成", "P2": "未开始"},
            "广告费": 0,
            "生产线数量": 1,
            "每季度固定支出": 10
        }

    def calculate_equity(self):
        equity = (self.data["现金"] + self.data["固定资产"]
                  + self.data["库存"] - self.data["贷款"])
        self.data["所有者权益"] = equity
        return equity

    def update_data(self, updates: dict):
        self.data.update(updates)
        self.calculate_equity()
        print("数据已更新，当前所有者权益：" + str(self.data["所有者权益"]))

    def show_status(self):
        print("\n" + "="*45)
        print("第 " + str(self.round) + " 轮财务状态")
        print("="*45)
        for k, v in self.data.items():
            print("  " + str(k) + ": " + str(v))
        print("="*45 + "\n")

    # ===== 功能1：现金流死线预测 =====
    def cash_flow_deadline(self):
        cash = self.data["现金"]
        fixed_cost = self.data["每季度固定支出"]
        loan = self.data["贷款"]
        loan_interest = loan * 0.05  # 假设5%利息
        total_cost_per_round = fixed_cost + loan_interest
        if total_cost_per_round <= 0:
            seasons = 999
        else:
            seasons = int(cash / total_cost_per_round)
        print("\n现金流预警：")
        print("  当前现金：" + str(cash))
        print("  每季度支出（含利息）：" + str(round(total_cost_per_round, 2)))
        print("  若拿不到订单，还能撑：" + str(seasons) + " 个季度")
        if seasons <= 2:
            print("  !!!高危警告：现金流紧张，谨慎贷款和广告!!!")
        return seasons

    # ===== 功能2：竞标压力分析 =====
    def competitor_analysis(self, competitor_ads: list):
        self.competitor_history.extend(competitor_ads)
        if len(self.competitor_history) == 0:
            print("暂无对手数据")
            return
        avg = sum(self.competitor_history) / len(self.competitor_history)
        max_ad = max(self.competitor_history)
        print("\n对手广告费分析：")
        print("  历史平均：" + str(round(avg, 1)))
        print("  历史最高：" + str(max_ad))
        print("  建议本轮广告费区间：" + str(round(avg * 0.9, 1)) + " ~ " + str(round(avg * 1.2, 1)))

    # ===== 功能3：视觉识别截图数据（Qwen3-VL）=====
    def extract_from_image(self, image_path: str):
        with open(image_path, "rb") as f:
            img_data = base64.b64encode(f.read()).decode("utf-8")
        prompt = "请提取图中所有订单的产品名、数量、单价，输出为Python字典格式。"
        resp = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen3-vl:8b",
                "prompt": prompt,
                "images": [img_data],
                "stream": False
            },
            timeout=120
        )
        result = resp.json()["response"]
        print("\n图像识别结果：")
        print(result)
        return result

    # ===== 功能4：主决策建议（Qwen3）=====
    def get_strategic_advice(self, market_info: str):
        self.show_status()
        seasons_left = self.cash_flow_deadline()

        prompt = (
            "你是ERP比赛CEO顾问，给出第" + str(self.round) + "轮决策建议。\n"
            "财务数据：" + str(self.data) + "\n"
            "市场信息：" + market_info + "\n"
            "历史决策：" + str(self.history[-3:] if self.history else "暂无") + "\n"
            "现金流警告：还能撑" + str(seasons_left) + "季度\n"
            "核心规则：最终以所有者权益高低决定胜负，贷款有利息风险。\n\n"
            "请给出：\n"
            "1. 广告费建议金额及理由\n"
            "2. 是否贷款，贷多少\n"
            "3. 生产线调整建议\n"
            "4. 研发投入建议\n"
            "5. 本轮最优先事项"
        )

        print("AI正在分析，请稍候...")
        resp = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": "qwen3:8b", "prompt": prompt, "stream": False},
            timeout=180
        )
        advice = resp.json()["response"]
        self.history.append({
            "轮次": self.round,
            "市场": market_info,
            "权益": self.data["所有者权益"]
        })
        self.round += 1
        return advice


# ========== 启动 ==========
my_agent = ERPAgent()

# 输入对手上轮广告费（没有就留空列表）
my_agent.competitor_analysis([15, 20, 18])

# 输入本轮市场信息，运行获取建议
market_info = "本轮市场需求旺盛，竞争对手加大广告投入，原材料价格上涨10%"
advice = my_agent.get_strategic_advice(market_info)
print("\nAI建议：\n")
print(advice)


对手广告费分析：
  历史平均：17.7
  历史最高：20
  建议本轮广告费区间：15.9 ~ 21.2

第 1 轮财务状态
  现金: 100
  固定资产: 50
  库存: 0
  贷款: 0
  所有者权益: 100
  研发进度: {'P1': '完成', 'P2': '未开始'}
  广告费: 0
  生产线数量: 1
  每季度固定支出: 10


现金流预警：
  当前现金：100
  每季度支出（含利息）：10.0
  若拿不到订单，还能撑：10 个季度
AI正在分析，请稍候...


In [ ]:
# 基于“赛项 A 卷”官方数据的参数初始化
params = {
    # --- 初始财务状态 ---
    "初始现金": 100000,           # 现金 100,000
    "初始所有者权益": 700000,      # 所有者权益合计 700,000
    "原材料价值": 600000,         # R1-R4 各 300 个，合计 600,000
    
    # --- 运营硬性规则 ---
    "预算使用率下限": 0.8,         # 低于 80% 扣 10000 分
    "预算使用率上限": 1.2,         # 高于 120% 扣 10000 分
    "所得税率": 0.25,             # 25%
    "违约扣款比例": 0.40,          # 违约扣 40%
    
    # --- 费用类 ---
    "每季管理费": 10000,          # 固定管理费 10,000
    "折旧年限_传统": 1,            # 传统线 1 年折旧
    "折旧年限_全自动": 3,          # 全自动 3 年折旧
    
    # --- 碳中和 ---
    "初始碳排放量": 100000,        # 100,000 吨
    "碳中和单价": 20              # 20 元/吨
}

In [13]:
import pandas as pd
import os

# 自动获取桌面路径
desktop = os.path.join(os.path.expanduser('\~'), 'Desktop')
file_name = '经销商订单明细 (1).xls'   # ← 确认文件名完全一致

file_path = os.path.join(desktop, file_name)

# 关键修复：指定 engine='xlrd'（专门用于老版 .xls 文件）
df = pd.read_excel(file_path, engine='xlrd')

print("✅ 读取成功！")
print("数据行数:", len(df))
print("列名:", df.columns.tolist())
print("\n前5行预览:")
print(df.head())

<>:5: SyntaxWarning: "\~" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\~"? A raw string is also an option.
<>:5: SyntaxWarning: "\~" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\~"? A raw string is also an option.
C:\Users\Microsoft\AppData\Local\Temp\ipykernel_23772\2315238066.py:5: SyntaxWarning: "\~" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\~"? A raw string is also an option.
  desktop = os.path.join(os.path.expanduser('\~'), 'Desktop')


ImportError: `Import xlrd` failed. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install the xlrd package.

In [ ]:
import pandas as pd

# 读取订单表
df_orders = pd.read_csv('经销商订单明细 (1).xls - sheet1.csv')

def get_best_order(year, quarter):
    # [span_17](start_span)筛选当前季度的订单[span_17](end_span)
    current = df_orders[(df_orders['年份'] == year) & (df_orders['季度'] == quarter)].copy()
    
    # 简易利润估算：参考价 - P1成本(2000) - 贴现损失(按8%估算)
    current['估算毛利'] = current['供应商参考价格(元)'] - 2000 - (current['供应商参考价格(元)'] * 0.08)
    
    # 返回利润最高的前 5 个订单
    return current.sort_values(by='估算毛利', ascending=False).head(5)

# 测试一下第 1 年第 2 季度的订单
print(get_best_order(1, 2))

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# 解决中文显示问题
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False

# 读取订单
df = pd.read_csv('经销商订单明细 (1).xls - sheet1.csv')

# 筛选 1 年 2 季度的订单并简单算个利润（参考价 - P1成本2000）
q2_orders = df[(df['年份'] == 1) & (df['季度'] == 2)].copy()
q2_orders['估算利润'] = q2_orders['供应商参考价格(元)'] - 2000

# 画图
q2_orders.plot(kind='bar', x='编号', y='估算利润', color='skyblue', figsize=(10, 5))
plt.title('1年2季度各订单利润预测')
plt.ylabel('利润 (元)')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '经销商订单明细 (1).xls - sheet1.csv'

In [3]:
import openai

# 1. 建立与本地 Ollama 的连接
client = openai.OpenAI(
    base_url='http://localhost:11434/v1/',  # 注意：最后这个 /v1/ 绝对不能漏掉
    api_key='ollama',                       # 随便填，但绝不能是空的
)

# 2. 呼叫你的 DeepSeek 大脑
try:
    response = client.chat.completions.create(
        model="deepseek-r1:8b",             # 这里的名字必须和你黑框框里的一模一样
        messages=[
            {"role": "user", "content": "你好，我是沙盘比赛企业的CEO。测试通讯，收到请用中文回复一句简短的确认。"}
        ]
    )
    # 3. 打印模型回复
    print("🤖 Agent 回复：")
    print(response.choices[0].message.content)

except Exception as e:
    print(f"❌ 还是报错了，错误信息是：\n{e}")

🤖 Agent 回复：
确认收到此测试。如有其他事宜，随时沟通。
